# Many Models Forecasting Demo

This notebook showcases how to run MMF with MLForecast-based global models on multiple time series of daily resolution. We use [`MLForecastLGBM`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/mlforecast/MLForecastPipeline.py) (LightGBM with fixed hyperparameters) and [`MLForecastAutoLGBM`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/mlforecast/MLForecastPipeline.py) (LightGBM with Optuna-driven hyperparameter and feature tuning), both built on top of [MLForecast](https://nixtlaverse.nixtla.io/mlforecast/index.html). We will use [M4 competition](https://www.sciencedirect.com/science/article/pii/S0169207019301128#sec5) daily data.

Unlike the [`global_daily_dl.ipynb`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/examples/daily/global_daily_dl.ipynb) notebook (which trains GPU-bound neural forecasters on A10G), this notebook stays on CPU and runs end-to-end in a single Python process — no `dbutils.notebook.run` iteration needed, since LightGBM does not exhibit the GPU-memory accumulation that motivates that pattern.

### Cluster setup

We recommend using a **single-node CPU cluster** with [Databricks Runtime 18 for ML](https://docs.databricks.com/aws/en/release-notes/runtime/18ml). The pinned versions in [`requirements-global.txt`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/requirements-global.txt) (`mlforecast==1.0.31`, `lightgbm==4.6.0`, `optuna==3.6.1`) match the DBR 18 ML preinstalled versions exactly, so the install is a near-no-op.

A machine like `m5d.8xlarge` on AWS (32 vCPU / 128 GB RAM) or `Standard_D32ds_v5` on Azure is a good starting point. The whole pipeline (data load → fit → backtest → score) runs on the driver in this CPU integration, so multi-node clusters do not provide additional parallelism here. The driver-only path supports panels up to a few million `(unique_id, ds)` rows comfortably.

### Install and import packages
Check out [requirements-global.txt](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/requirements-global.txt) if you're interested in the libraries we use.

In [ ]:
%pip install -r ../../requirements-global.txt --quiet
%pip install datasetsforecast==0.0.8 --quiet
dbutils.library.restartPython()

In [ ]:
import logging
import pathlib
import pandas as pd
from datasetsforecast.m4 import M4
from mmf_sa import run_forecast

logging.getLogger("py4j.clientserver").setLevel(logging.WARNING)
logging.getLogger("py4j.java_gateway").setLevel(logging.WARNING)

### Prepare data
We are using [`datasetsforecast`](https://github.com/Nixtla/datasetsforecast/tree/main/) package to download M4 data. M4 dataset contains a set of time series which we use for testing MMF. Below we have written a number of custom functions to convert M4 time series to an expected format.

In [ ]:
# Number of time series
n = 100


def create_m4_daily():
    y_df, _, _ = M4.load(directory=str(pathlib.Path.home()), group="Daily")
    target_ids = {f"D{i}" for i in range(1, n)}
    y_df = y_df[y_df["unique_id"].isin(target_ids)]
    y_df = (
        y_df.groupby("unique_id", group_keys=False)
             .apply(lambda g: transform_group(g, g.name))
             .reset_index(drop=True)
    )
    return y_df


def transform_group(df, unique_id):
    if len(df) > 1020:
        df = df.iloc[-1020:]
    start = pd.Timestamp("2020-01-01")
    date_idx = pd.date_range(start=start, periods=len(df), freq="D", name="ds")
    res_df = pd.DataFrame({
        "ds": date_idx,
        "unique_id": unique_id,
        "y": df["y"].to_numpy()
    })
    return res_df

We are going to save this data in a delta lake table. Provide catalog and database names where you want to store the data.

In [ ]:
catalog = "mmf"  # Name of the catalog we use to manage our assets
db = "m4"  # Name of the schema we use to manage our assets (e.g. datasets)
user = spark.sql('select current_user() as user').collect()[0]['user']  # User email address

In [ ]:
# Making sure that the catalog and the schema exist
_ = spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
_ = spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{db}")

(
    spark.createDataFrame(create_m4_daily())
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"{catalog}.{db}.m4_daily_train")
)

Let's take a peek at the dataset:

In [ ]:
display(
  spark.sql(f"select * from {catalog}.{db}.m4_daily_train where unique_id in ('D1', 'D2', 'D3', 'D4', 'D5') order by unique_id, ds")
  )

### Models

We will run two MLForecast-based CPU global models:

- **`MLForecastLGBM`** — LightGBM trained with a fixed set of hyperparameters and feature pipeline (lags, date features, target/lag transforms). Use this when you have a known-good configuration and want fast, deterministic training.
- **`MLForecastAutoLGBM`** — Same model wrapped in MLForecast's `AutoMLForecast` + Optuna, so LightGBM hyperparameters **and** the feature pipeline (lags, date features, target transforms, lag transforms, optional `static_features` toggle) are tuned jointly via cross-validated rolling-origin trials.

Both models are configured in [`mmf_sa/models/models_conf.yaml`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/models_conf.yaml) and have daily-frequency-specific defaults in [`mmf_sa/forecasting_conf_daily.yaml`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/forecasting_conf_daily.yaml). The search spaces (`model_hp_space`, `feature_space`, `fit_space`) for the Auto variant are fully YAML-driven — modify them there to adjust the HPO surface.

In [ ]:
active_models = [
    "MLForecastLGBM",
    "MLForecastAutoLGBM",
]

### Run MMF

Now we run evaluation and forecasting using `run_forecast` (defined in [`mmf_sa/__init__.py`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/__init__.py)). Refer to [README.md](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/README.md#parameters-description) for a comprehensive description of each parameter.

Note that we are not providing any covariate field (`static_features`, `dynamic_future_*`, `dynamic_historical_*`) in this example. See [`examples/external_regressors/global_external_regressors_daily_dl.ipynb`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/examples/external_regressors/global_external_regressors_daily_dl.ipynb) for the pattern with exogenous regressors — `MLForecastLGBM`/`MLForecastAutoLGBM` support all four covariate types via the same MMF keys.

Set `accelerator="cpu"` to dispatch to the CPU global-model path. Unlike the GPU global notebook, we pass **both models** to `active_models` in a single call — LightGBM doesn't accumulate GPU memory between fits, so we don't need to iterate via `dbutils.notebook.run`.

While the cell below runs you can monitor progress in the MLflow Experiments view at the `experiment_path` you provide. One run per model is logged with hyperparameters and the aggregated metric.

In [ ]:
run_forecast(
    spark=spark,
    train_data=f"{catalog}.{db}.m4_daily_train",
    scoring_data=f"{catalog}.{db}.m4_daily_train",
    scoring_output=f"{catalog}.{db}.daily_scoring_output",
    evaluation_output=f"{catalog}.{db}.daily_evaluation_output",
    model_output=f"{catalog}.{db}",
    group_id="unique_id",
    date_col="ds",
    target="y",
    freq="D",
    prediction_length=10,
    backtest_length=30,
    stride=10,
    metric="smape",
    train_predict_ratio=1,
    data_quality_check=True,
    resample=False,
    active_models=active_models,
    experiment_path=f"/Users/{user}/mmf/m4_daily",
    use_case_name="m4_daily",
    accelerator="cpu",
)

### Evaluate

The `evaluation_output` table stores all evaluation results for all backtesting trials from all models, including any other models you may have already run (local, GPU global, foundation) with the same `use_case_name`. This makes it easy to compare CPU LightGBM models against GPU neural forecasters or local statistical models side-by-side.

In [ ]:
display(spark.sql(f"""
    select * from {catalog}.{db}.daily_evaluation_output
    where unique_id = 'D1' and model in ('MLForecastLGBM', 'MLForecastAutoLGBM')
    order by unique_id, model, backtest_window_start_date
    """))

For global models we train the model once using the training dataset excluding `backtest_length`. We then use the same fitted model to produce as-if forecasts for all backtesting periods. This avoids data leakage and matches how MMF backtests other global models. See `backtest` in [`mmf_sa/models/abstract_model.py`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/abstract_model.py) and the per-iteration `predict` logic in [`mmf_sa/models/mlforecast/MLForecastPipeline.py`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/mlforecast/MLForecastPipeline.py) (which uses MLForecast's `new_df` argument to re-align the per-series prediction window each iteration).

The model that gets registered in Unity Catalog (`mmf.m4.MLForecastLGBM_m4_daily`, `mmf.m4.MLForecastAutoLGBM_m4_daily`) is trained on the **full** history including the backtest window. The `model_uri` column in the evaluation table points at the corresponding MLflow run.

### Forecast

The `scoring_output` table stores per-series forecasts (`prediction_length` steps ahead from the last observed date) for each model. Use this for downstream consumption.

In [ ]:
display(spark.sql(f"""
    select * from {catalog}.{db}.daily_scoring_output
    where unique_id = 'D1' and model in ('MLForecastLGBM', 'MLForecastAutoLGBM')
    order by unique_id, model, ds
    """))

Refer to the [post-evaluation analysis notebook](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/examples/post-evaluation-analysis.ipynb) for guidance on selecting a winning model (or building an ensemble) across all model families you've run against the same `use_case_name`.

### Delete Tables
Let's clean up the tables.

In [ ]:
#display(spark.sql(f"delete from {catalog}.{db}.daily_evaluation_output where model in ('MLForecastLGBM', 'MLForecastAutoLGBM')"))

In [ ]:
#display(spark.sql(f"delete from {catalog}.{db}.daily_scoring_output where model in ('MLForecastLGBM', 'MLForecastAutoLGBM')"))